# Foundation 04 — Secure Tool and Action Interface Design

Design a narrow versioned tool contract, reject broad or stale interfaces, distinguish structural validation from business authorization, validate results before model exposure, and reconcile an unknown provider outcome without duplicating an effect.

![Secure tool proposal, gateway, result admission, and recovery](architecture.svg)

The model and external provider are both untrusted boundaries. An application-owned gateway resolves the exact contract, combines trusted identity and state, dispatches one allowlisted adapter, and records bounded evidence.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
from dataclasses import replace
import sys
sys.path.insert(0, '.')
ns = runpy.run_path('lab.py')
ToolProposal, DecisionStatus, ProviderMode = (ns[name] for name in ('ToolProposal','DecisionStatus','ProviderMode'))
build_gateway, refund_proposal, issue_demo_grant = (ns[name] for name in ('build_gateway','refund_proposal','issue_demo_grant'))
now = datetime(2026,9,22,12,0,tzinfo=timezone.utc)
gateway, actor = build_gateway()

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
proposal = refund_proposal('notebook:refund:1',2500)
grant = issue_demo_grant(proposal,actor,grant_id='grant:notebook:1',now=now)
gateway.register_grant(grant)
allowed = gateway.dispatch(actor,proposal,grant_id=grant.grant_id,now=now)
assert allowed.effect_applied and allowed.reason == 'contract-allow'
{'decision': allowed, 'provider_calls': gateway.provider.calls, 'trace': gateway.traces[-1]}

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
broad = ToolProposal('run_shell','1.0.0',{'command':'refund --all'},'notebook:attack')
baseline_accepts = ns['unsafe_broad_dispatch'](broad)
controlled = gateway.dispatch(actor,broad,now=now)
assert baseline_accepts and controlled.reason == 'tool-unregistered'
{'unsafe_baseline': baseline_accepts, 'controlled': controlled}

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
extra = replace(proposal,logical_operation_id='notebook:extra',arguments={**proposal.arguments,'is_admin':True})
boolean = refund_proposal('notebook:boolean',True)
extra_grant = issue_demo_grant(extra,actor,grant_id='grant:extra',now=now)
bool_grant = issue_demo_grant(boolean,actor,grant_id='grant:boolean',now=now)
gateway.register_grant(extra_grant); gateway.register_grant(bool_grant)
extra_result = gateway.dispatch(actor,extra,grant_id=extra_grant.grant_id,now=now)
bool_result = gateway.dispatch(actor,boolean,grant_id=bool_grant.grant_id,now=now)
assert extra_result.reason == 'unknown-field' and bool_result.reason == 'field-type'
(extra_result,bool_result)

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
report,cases = ns['evaluate_controls'](now=now)
assert (report.cases,report.valid_cases,report.attack_cases,report.failure_cases) == (12,3,7,2)
assert report.attack_block_rate == report.valid_task_success_rate == 1
assert report.attack_effect_rate == 0 and report.unsafe_baseline_attack_acceptance_rate == 1
{'populations': {'all':report.cases,'valid':report.valid_cases,'attack':report.attack_cases,'failure':report.failure_cases}, 'rates': {'attack_block':report.attack_block_rate,'attack_effect':report.attack_effect_rate,'valid_success':report.valid_task_success_rate,'baseline_attack_acceptance':report.unsafe_baseline_attack_acceptance_rate,'trace_completeness':report.trace_completeness_rate}, 'outcomes': {case.name:(case.decision.status.value,case.decision.reason) for case in cases}}

## 6. Exercise a second failure mode

In [ ]:
unknown_gateway, unknown_actor = build_gateway(provider_mode=ProviderMode.TIMEOUT_AFTER_COMMIT)
unknown_proposal = refund_proposal('notebook:unknown',500)
unknown_grant = issue_demo_grant(unknown_proposal,unknown_actor,grant_id='grant:unknown',now=now)
unknown_gateway.register_grant(unknown_grant)
unknown = unknown_gateway.dispatch(unknown_actor,unknown_proposal,grant_id=unknown_grant.grant_id,now=now)
recovered = unknown_gateway.dispatch(unknown_actor,unknown_proposal,grant_id=unknown_grant.grant_id,now=now+timedelta(seconds=1))
assert unknown.status is DecisionStatus.UNKNOWN
assert recovered.reason == 'reconciled-confirmed' and unknown_gateway.provider.calls == 1
(unknown,recovered)

## 7. Schema-valid is not authorized

The cross-tenant request passes structural validation but fails current business and identity state before dispatch.

In [ ]:
cross = ToolProposal('issue_approved_refund','1.0.0',{'case_id':'case:south:9','amount_cents':500,'currency':'CAD'},'notebook:tenant')
cross_grant = issue_demo_grant(cross,actor,grant_id='grant:tenant',now=now)
gateway.register_grant(cross_grant)
denied = gateway.dispatch(actor,cross,grant_id=cross_grant.grant_id,now=now)
assert denied.reason == 'tenant' and not denied.effect_applied
denied

## 8. Tool output is another trust boundary

A malformed provider result is rejected before it can enter model context.

In [ ]:
bad_gateway,bad_actor = build_gateway(provider_mode=ProviderMode.MALFORMED_OUTPUT)
bad = refund_proposal('notebook:output',500)
bad_grant = issue_demo_grant(bad,bad_actor,grant_id='grant:output',now=now)
bad_gateway.register_grant(bad_grant)
blocked = bad_gateway.dispatch(bad_actor,bad,grant_id=bad_grant.grant_id,now=now)
assert blocked.status is DecisionStatus.ERROR and blocked.result is None
blocked

## 9. OpenAI Agents SDK and Pydantic: real contract mapping

The pinned libraries generate strict input and output schemas and expose a workflow approval hook. The application gateway still owns authorization and dispatch; no model or network call is made.

In [ ]:
sdk = runpy.run_path('sdk_adapter.py')
sdk_decision,evidence = sdk['credential_free_demo'](now=now)
model = evidence['input_schema']['$defs']['IssueApprovedRefundInput']
assert sdk_decision.effect_applied and evidence['strict_json_schema'] and evidence['needs_approval']
assert model['additionalProperties'] is False and evidence['output_schema']['additionalProperties'] is False
{'decision':sdk_decision,'fields':sorted(model['properties']),'timeout_seconds':evidence['timeout_seconds']}

## 10. Production replacement

Production replacement: authenticated user and workload identity; an immutable owned schema registry with provenance and lifecycle; current resource authorization immediately before dispatch; exact revocable authorization evidence; allowlisted adapters; durable idempotency and concurrency control; provider idempotency and lookup; reconciliation queues and operator workflows; rate limits, deadlines, circuit breakers, and safe degradation; input/output size and classification controls; protected traces; continuous contract, concurrency, chaos, and adversarial tests; and tested kill switches. The local dictionaries, lock, synthetic grant, and provider simulator prove only the in-process control sequence.

## 11. Exercises

1. Add a narrow case-summary read with a result-size limit.
2. Retire a contract version and prove stale discovery fails closed.
3. Contrast timeout before dispatch with timeout after commit.
4. Add semantic output binding for case ID and amount.
5. Replace the local operation table with a durable uniqueness constraint.
6. Map the contract to MCP or another SDK while preserving application-owned authorization.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.